# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kenzo4k/Flyrank-ML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected lane:** Lane 4 — CTR / Engagement Opportunity Scoring

**Why this lane?**
Pages with high impressions (`impressions_90d`) but low CTR relative to their position tier are latent opportunities — they have already earned Google's trust (visible rankings) but are losing organic clicks to weaker titles, mismatched meta descriptions, or poor snippet structure. The fix is copywriting-level, not a full content rebuild.

CTR opportunity is measurable: the gap between a page's observed CTR and the median CTR for its position cohort (e.g. `top_3`, `page_1`, `striking`) translates directly into estimated missing clicks = impressions × gap. This gives editors a ranked queue with a clear, quantified upside per item.

> **Note on visibility signal:** Lane justification uses `impressions_90d` (observed search impressions from GSC) as the visibility measure — not `search_volume`, which is a keyword-level monthly estimate that does not aggregate across the many long-tail queries a single page may rank for.

In [1]:
import os, sys, subprocess
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kenzo4k/Flyrank-ML"
REPO_DIR = "Flyrank-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

data_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv"
]
data_path = next((p for p in data_candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded: 30,000 rows x 44 columns


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision to improve:** Which visible pages (already earning impressions in GSC) are significantly underperforming their position-tier CTR benchmark and should be prioritised for title, meta description, and snippet optimisation?

**Who acts on the output?** SEO copywriters, CRO strategists, and digital marketers — they allocate limited editing hours to the pages with the largest recoverable click volume.

**Cost of a wrong call:**
- *False positive* — rewriting a title on a page whose CTR is already at or above its tier benchmark. Risk: destabilising a well-performing snippet, wasting editor time.
- *False negative* — ignoring a high-impression page with a genuine CTR gap. Cost: leaving measurable organic clicks on the table despite holding Page 1 or Top 3 rankings.

**Why ML / data helps over fixed rules:** CTR decays non-linearly with position. A page at rank 8 and a page at rank 2 cannot be judged by the same threshold. Position-adjusted residual benchmarking isolates true underperformers. An ML model can further condition on intent, content type, and competition to sharpen the ranking beyond what a single static rule can achieve.

## 3. Quick look at the data (2–3 real numbers)

*Load the starter CSV and show 2–3 real numbers that make your lane look worth the next 7 weeks.*

**Preprocessing order (critical — fixes the benchmark bug):**
1. Filter `avg_position > 0` — rows where `avg_position == 0` mean "no ranking data" per the data dictionary, NOT rank zero. Including them pollutes every tier median.
2. Filter `impressions_90d >= 100` — removes near-zero-impression noise rows before computing medians.
3. **Then** compute median CTR per `position_tier`.

In [6]:
df_valid = df[(df['avg_position'] > 0) & (df['impressions_90d'] >= 100)].copy()

# Compute tier benchmarks on clean, volume-filtered data
ctr_benchmarks = df_valid.groupby('position_tier').agg(
    total_pages=('content_id', 'count'),
    total_impressions=('impressions_90d', 'sum'),
    total_clicks=('clicks_90d', 'sum'),
    median_ctr=('ctr', 'median')
).reset_index()

ctr_benchmarks['weighted_ctr_pct'] = (
    ctr_benchmarks['total_clicks'] / ctr_benchmarks['total_impressions'] * 100
).round(2)

print("--- Position Tier CTR Benchmarks (filtered: avg_position > 0, impressions >= 100) ---")
print(ctr_benchmarks.to_string(index=False))

--- Position Tier CTR Benchmarks (filtered: avg_position > 0, impressions >= 100) ---
position_tier  total_pages  total_impressions  total_clicks  median_ctr  weighted_ctr_pct
         deep          879            1213203           479        0.00              0.04
       page_1         8633           89493618        313097        0.23              0.35
     page_3_5         6058           35141017         54370        0.06              0.15
     striking         5903           22946217         79584        0.15              0.35
        top_3          533            7025180         34222        0.19              0.49


1. **Merge Benchmarks:** Merge median tier CTR back into candidate dataframe.
2. **Compute Residual Gap:** Calculate `ctr_gap = median_tier_ctr - actual_ctr`.
3. **Extract 3 real numbers:** Report total underperforming candidates, estimated missing click potential, and breakdown by `content_type`.

In [5]:
# tier_benchmarks already computed on filtered data (df_valid)
tier_benchmarks = df_valid.groupby('position_tier')['ctr'].median().reset_index()
tier_benchmarks.columns = ['position_tier', 'median_tier_ctr']

# Merge benchmarks back onto the already-filtered candidate set
candidates = df_valid.merge(tier_benchmarks, on='position_tier')
candidates['ctr_gap'] = candidates['median_tier_ctr'] - candidates['ctr']

# Opportunity score: impressions × gap (only positive gaps = true underperformers)
candidates['opportunity_score'] = candidates['impressions_90d'] * (
    candidates['ctr_gap'].clip(lower=0) / 100
)

underperformers = candidates[candidates['ctr_gap'] > 0]

# ── The 3 real numbers ──────────────────────────────────────────────────────
total_underperformers = len(underperformers)
total_opportunity_clicks = underperformers['opportunity_score'].sum()

print(f"1. Total underperforming candidates: {total_underperformers:,}")
print(f"2. Estimated upper-bound click opportunity (directional): {total_opportunity_clicks:,.0f} clicks")
print()

content_breakdown = underperformers.groupby('content_type').agg(
    underperforming_pages=('content_id', 'count'),
    opportunity_score=('opportunity_score', 'sum')
)
content_breakdown['opportunity_score'] = content_breakdown['opportunity_score'].round(0)
print("3. Breakdown by content_type:")
print(content_breakdown)

print()
print("--- Tier-level underperformer summary ---")
tier_summary = underperformers.groupby('position_tier').agg(
    underperforming_pages=('content_id', 'count'),
    opportunity_score=('opportunity_score', 'sum')
).round(0)
print(tier_summary)

1. Total underperforming candidates: 10,307
2. Estimated upper-bound click opportunity (directional): 52,243 clicks

3. Breakdown by content_type:
                    underperforming_pages  opportunity_score
content_type                                                
comparison article                    284              255.0
feedly article                        160              204.0
keyword article                      9863            51783.0

--- Tier-level underperformer summary ---
               underperforming_pages  opportunity_score
position_tier                                          
page_1                          4262            41115.0
page_3_5                        2914             3616.0
striking                        2867             5396.0
top_3                            264             2116.0


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What this research CAN claim:**
- Observed historical position-adjusted CTR underperformance compared to cohort medians.
- A prioritised candidate list for title, meta description, and snippet optimisation.
- Estimated upper-bound click gap opportunity (directional, non-causal — actual gains depend on query intent mix, SERP layout, and competitor behaviour).

**What this research CANNOT claim:**
- Guaranteed CTR increases upon rewrites (CTR is influenced by brand recognition, query intent mix, SERP layout features, and competitor ad placements).
- Causal proof that low CTR causes ranking drops.
- Predicting exact ranking changes from content edits.

## Lane 4 Analysis: Conclusion

### Corrected Key Numbers (after filtering avg_position > 0 AND impressions ≥ 100 before tier medians)

| Position Tier | Median CTR | Underperforming Pages | Opportunity Score (clicks) |
|:---|:---:|:---:|:---:|
| `top_3` | ~0.19% | ~264 | ~2,116 |
| `page_1` | ~0.23% | ~4,262 | ~41,115 |
| `striking` | ~0.15% | ~2,867 | ~5,396 |
| `page_3_5` | ~0.06% | ~2,914 | ~3,616 |
| `deep` | ~0.00% | 0 | 0 |
| **TOTAL** | — | **~10,307** | **~52,243** |

> Numbers above are estimated from the corrected logic. The code cell above computes exact values on your run.

### What the Top 3 result actually means

In the original notebook, `top_3` showed a **0.00% median CTR** — this was a **data noise artifact**, not a real signal. The ~1,205 rows with `avg_position == 0` ("no data" sentinel) were silently bucketed into `top_3` (because `avg_position <= 3` caught `avg_position = 0`), dragging the median to zero. After correct filtering, `top_3` median CTR is ~0.19%, and ~264 pages have recoverable opportunity.

### Recommendation

**Primary focus: Page 1 keyword articles** — largest absolute opportunity (~41K clicks in the corrected estimate).

**Optimisation actions (beyond title/meta):**
- Title tag and meta description rewrites.
- Structured snippet improvements (FAQ schema, How-To markup).
- Intent-match review: ensure headline matches dominant query intent.
- On-page engagement improvements (time-on-page, scroll depth) where engagement signals are low.
- Monitoring: track CTR weekly for 4–6 weeks post-edit before claiming improvement.

**Estimated upper-bound click opportunity: ~52,243 clicks (directional, not guaranteed).**
This is a mathematical ceiling assuming every underperformer reaches its tier median exactly — actual results will vary.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.